# 04 — Galaxy Colours and Photometric Redshifts

The "colours" part of the project title. We extract multi-band CModel
fluxes from the DP0.2 catalog, compute colours, and examine the
photometric redshifts produced by the pipeline.

**Run this on the Rubin Science Platform (RSP).**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from lsst.daf.butler import Butler

In [ ]:
butler = Butler('dp02', collections='2.2i/runs/DP0.2')

tract = 4226
patches = [16, 17, 18, 23, 24, 25]

all_obj = []
for patch in patches:
    try:
        obj = butler.get('objectTable', tract=tract, patch=patch)
        all_obj.append(obj)
    except Exception:
        pass

df = pd.concat(all_obj, ignore_index=True)
print(f"Total objects: {len(df)}")

# Galaxy selection
gal_mask = (df['detect_isPrimary'] & (df['refExtendedness'] == 1))
gals = df[gal_mask].copy()
print(f"Galaxies: {len(gals)}")

## 1. Extract Multi-band Photometry

In [ ]:
# CModel fluxes in each band
bands = ['u', 'g', 'r', 'i', 'z', 'y']

# Find the flux column naming convention
flux_cols = {}
fluxerr_cols = {}
for b in bands:
    candidates = [c for c in gals.columns
                  if c.startswith(f'{b}_') and 'cmodel' in c.lower() and 'flux' in c.lower()
                  and 'err' not in c.lower() and 'flag' not in c.lower()]
    if candidates:
        flux_cols[b] = candidates[0]
        err_candidates = [c for c in gals.columns
                          if c.startswith(f'{b}_') and 'cmodel' in c.lower() and 'fluxerr' in c.lower()]
        if err_candidates:
            fluxerr_cols[b] = err_candidates[0]

print("Flux columns found:")
for b, col in flux_cols.items():
    print(f"  {b}: {col}")

In [ ]:
# Compute AB magnitudes from nJy fluxes
# m_AB = -2.5 * log10(f_nJy) + 31.4

mags = {}
mag_errs = {}

for b in bands:
    if b in flux_cols:
        flux = gals[flux_cols[b]].values
        with np.errstate(divide='ignore', invalid='ignore'):
            mag = -2.5 * np.log10(np.maximum(flux, 1e-30)) + 31.4
        mag[flux <= 0] = np.nan
        mags[b] = mag

        if b in fluxerr_cols:
            fluxerr = gals[fluxerr_cols[b]].values
            snr = flux / np.maximum(fluxerr, 1e-30)
            mag_errs[b] = 2.5 / (np.log(10) * np.maximum(snr, 0.1))

# Show magnitude distributions
fig, ax = plt.subplots(figsize=(10, 6))
colors_plot = {'u': 'purple', 'g': 'blue', 'r': 'green',
               'i': 'orange', 'z': 'red', 'y': 'darkred'}

for b in bands:
    if b in mags:
        valid = np.isfinite(mags[b]) & (mags[b] < 30)
        ax.hist(mags[b][valid], bins=60, range=(18, 28),
                histtype='step', lw=2, color=colors_plot[b],
                label=f'{b} (median={np.nanmedian(mags[b][valid]):.1f})')

ax.set_xlabel('AB Magnitude', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('CModel magnitude distributions (ugrizy)', fontsize=14)
ax.legend(fontsize=10)
ax.set_xlim(18, 28)
plt.tight_layout()
plt.savefig('../../figures/magnitude_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Colour-Colour Diagrams

In [ ]:
# Compute colours
gr = mags.get('g', np.full(len(gals), np.nan)) - mags.get('r', np.full(len(gals), np.nan))
ri = mags.get('r', np.full(len(gals), np.nan)) - mags.get('i', np.full(len(gals), np.nan))
iz = mags.get('i', np.full(len(gals), np.nan)) - mags.get('z', np.full(len(gals), np.nan))
ug = mags.get('u', np.full(len(gals), np.nan)) - mags.get('g', np.full(len(gals), np.nan))

# SNR cut for clean colours
i_mag = mags.get('i', np.full(len(gals), np.nan))
bright = np.isfinite(i_mag) & (i_mag < 25.0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# g-r vs r-i
ax = axes[0]
good = bright & np.isfinite(gr) & np.isfinite(ri)
ax.hexbin(ri[good], gr[good], gridsize=60, cmap='Blues', mincnt=1,
          extent=(-0.5, 1.5, -0.5, 2.0))
ax.set_xlabel('$r - i$', fontsize=12)
ax.set_ylabel('$g - r$', fontsize=12)
ax.set_title('$g-r$ vs $r-i$', fontsize=13)

# r-i vs i-z
ax = axes[1]
good = bright & np.isfinite(ri) & np.isfinite(iz)
ax.hexbin(iz[good], ri[good], gridsize=60, cmap='Greens', mincnt=1,
          extent=(-0.5, 1.0, -0.5, 1.5))
ax.set_xlabel('$i - z$', fontsize=12)
ax.set_ylabel('$r - i$', fontsize=12)
ax.set_title('$r-i$ vs $i-z$', fontsize=13)

# u-g vs g-r
ax = axes[2]
good = bright & np.isfinite(ug) & np.isfinite(gr)
ax.hexbin(gr[good], ug[good], gridsize=60, cmap='Oranges', mincnt=1,
          extent=(-0.5, 2.0, -0.5, 3.0))
ax.set_xlabel('$g - r$', fontsize=12)
ax.set_ylabel('$u - g$', fontsize=12)
ax.set_title('$u-g$ vs $g-r$', fontsize=13)

plt.suptitle('Colour-Colour Diagrams (CModel photometry)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../figures/colour_colour.png', dpi=150, bbox_inches='tight')
plt.show()

print("The bimodality in g-r vs r-i separates the red sequence")
print("(ellipticals, top-left) from the blue cloud (spirals, bottom-right).")
print("These colour sequences shift with redshift — the basis for photo-z.")

## 3. Colour vs. Redshift

Match to truth to see how colours track redshift.

In [ ]:
# Query truth redshifts via TAP
from lsst.rsp import get_tap_service

service = get_tap_service('tap')

query = """
SELECT
    obj.objectId,
    obj.g_cModelFlux, obj.r_cModelFlux, obj.i_cModelFlux,
    obj.z_cModelFlux, obj.y_cModelFlux,
    truth.redshift
FROM
    dp02_dc2_catalogs.Object AS obj
JOIN
    dp02_dc2_catalogs.MatchesTruth AS mt ON obj.objectId = mt.objectId
JOIN
    dp02_dc2_catalogs.TruthSummary AS truth ON mt.id_truth_type = truth.id_truth_type
WHERE
    obj.detect_isPrimary = 1
    AND obj.refExtendedness = 1
    AND obj.tract = 4226
    AND truth.truth_type = 1
    AND obj.i_cModelFlux / obj.i_cModelFluxErr > 10
"""

try:
    results = service.search(query)
    matched = results.to_table().to_pandas()
    print(f"Matched galaxies with redshifts: {len(matched)}")
    has_truth = True
except Exception as e:
    print(f"TAP query failed: {e}")
    print("Adapt table/column names to your DP0.2 schema.")
    has_truth = False

In [ ]:
if has_truth:
    # Compute colours from matched catalog
    def flux_to_mag(flux):
        with np.errstate(divide='ignore', invalid='ignore'):
            m = -2.5 * np.log10(np.maximum(flux, 1e-30)) + 31.4
        m[flux <= 0] = np.nan
        return m

    z_truth = matched['redshift'].values
    gr_matched = flux_to_mag(matched['g_cModelFlux']) - flux_to_mag(matched['r_cModelFlux'])
    ri_matched = flux_to_mag(matched['r_cModelFlux']) - flux_to_mag(matched['i_cModelFlux'])
    iz_matched = flux_to_mag(matched['i_cModelFlux']) - flux_to_mag(matched['z_cModelFlux'])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, colour, name, ylim in zip(axes,
            [gr_matched, ri_matched, iz_matched],
            ['$g - r$', '$r - i$', '$i - z$'],
            [(-0.5, 2.0), (-0.5, 1.5), (-0.5, 1.0)]):
        good = np.isfinite(colour) & np.isfinite(z_truth)
        ax.hexbin(z_truth[good], colour[good], gridsize=60, cmap='inferno',
                  mincnt=1, extent=(0, 2.5, ylim[0], ylim[1]))
        ax.set_xlabel('True redshift $z$', fontsize=12)
        ax.set_ylabel(name, fontsize=12)
        ax.set_title(f'{name} vs redshift', fontsize=13)

    plt.suptitle('Galaxy Colour vs. True Redshift', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../../figures/colour_vs_redshift.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("The 4000Å break moves through g→r at z~0.4, r→i at z~0.8, i→z at z~1.2")
    print("This is exactly what photo-z algorithms exploit.")
else:
    print("Skipping — no truth data available. Run on RSP with TAP access.")

## 4. Photo-z from DP0.2

DP0.2 includes photometric redshift estimates. Let's evaluate them.

In [ ]:
# Check if photo-z columns exist in the object table
photoz_cols = [c for c in df.columns if 'photoz' in c.lower() or 'photo_z' in c.lower()
               or 'zphot' in c.lower() or c == 'z_phot']
print(f"Photo-z columns: {photoz_cols}")

# If not in objectTable, try a separate dataset
if not photoz_cols:
    print("\nNo photo-z in objectTable. Checking for separate photo-z datasets...")
    pz_types = [dt for dt in butler.registry.queryDatasetTypes()
                if 'photoz' in dt.name.lower() or 'photo_z' in dt.name.lower()]
    print(f"Photo-z dataset types: {[dt.name for dt in pz_types]}")

In [ ]:
# If photo-z is available via TAP:
if has_truth:
    pz_query = """
    SELECT
        obj.objectId,
        obj.coord_ra, obj.coord_dec,
        truth.redshift AS z_true
    FROM
        dp02_dc2_catalogs.Object AS obj
    JOIN
        dp02_dc2_catalogs.MatchesTruth AS mt ON obj.objectId = mt.objectId
    JOIN
        dp02_dc2_catalogs.TruthSummary AS truth ON mt.id_truth_type = truth.id_truth_type
    WHERE
        obj.detect_isPrimary = 1
        AND obj.refExtendedness = 1
        AND obj.tract = 4226
        AND truth.truth_type = 1
        AND obj.i_cModelFlux / obj.i_cModelFluxErr > 20
    """
    # Note: add photo-z column if available in your DP0.2 schema
    # e.g., photo_z.z_mean, photo_z.z_median, etc.
    print("Adapt this query to include photo-z columns for your schema.")
    print("Then plot z_phot vs z_true as in notebook 02_pipeline/08_photoz.")

## 5. Colour-Shape Correlation

Do galaxy shapes correlate with colours? This is relevant because
**intrinsic alignments** — the tendency of galaxies to align with
their environment — depend on galaxy type (red vs. blue).

In [ ]:
# Find shape columns
e1_col = [c for c in gals.columns if 'e1' in c.lower() and 'hsm' in c.lower()
          and ('regauss' in c.lower() or 'Regauss' in c)]
e2_col = [c for c in gals.columns if 'e2' in c.lower() and 'hsm' in c.lower()
          and ('regauss' in c.lower() or 'Regauss' in c)]

if e1_col and e2_col:
    e1 = gals[e1_col[0]].values
    e2 = gals[e2_col[0]].values
    e_mag_gal = np.sqrt(e1**2 + e2**2)

    # Split by colour (red vs blue using g-r colour)
    gr_gal = (mags.get('g', np.full(len(gals), np.nan))
              - mags.get('r', np.full(len(gals), np.nan)))

    good = (np.isfinite(e_mag_gal) & np.isfinite(gr_gal)
            & (e_mag_gal < 1) & bright)

    red = good & (gr_gal > 0.8)
    blue = good & (gr_gal < 0.5)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # |e| distribution by colour
    ax = axes[0]
    ax.hist(e_mag_gal[red], bins=40, range=(0, 1), density=True,
            histtype='step', lw=2, color='red', label=f'Red (g-r>0.8, N={red.sum()})')
    ax.hist(e_mag_gal[blue], bins=40, range=(0, 1), density=True,
            histtype='step', lw=2, color='blue', label=f'Blue (g-r<0.5, N={blue.sum()})')
    ax.set_xlabel('$|e|$', fontsize=12)
    ax.set_ylabel('Normalized count', fontsize=12)
    ax.set_title('Ellipticity by galaxy colour', fontsize=13)
    ax.legend(fontsize=10)

    # <|e|> vs g-r colour
    ax = axes[1]
    gr_bins = np.linspace(-0.5, 2.0, 20)
    gr_centers = 0.5 * (gr_bins[:-1] + gr_bins[1:])
    mean_e = []
    err_e = []
    for i in range(len(gr_bins)-1):
        mask = good & (gr_gal >= gr_bins[i]) & (gr_gal < gr_bins[i+1])
        if mask.sum() > 10:
            mean_e.append(np.mean(e_mag_gal[mask]))
            err_e.append(np.std(e_mag_gal[mask]) / np.sqrt(mask.sum()))
        else:
            mean_e.append(np.nan)
            err_e.append(np.nan)

    ax.errorbar(gr_centers, mean_e, yerr=err_e, fmt='o-', color='steelblue',
                ms=6, capsize=3)
    ax.set_xlabel('$g - r$ colour', fontsize=12)
    ax.set_ylabel('$\\langle |e| \\rangle$', fontsize=12)
    ax.set_title('Mean ellipticity vs. colour', fontsize=13)

    plt.suptitle('Galaxy Colour-Shape Relation', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../../figures/colour_shape_relation.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("Red (elliptical) galaxies tend to be rounder than blue (spiral) galaxies.")
    print("This has implications for intrinsic alignment modeling in weak lensing.")
else:
    print("Shape columns not found — adapt column names.")

## Summary

This notebook completes the "shapes and colours" picture:

1. **CModel photometry** across ugrizy bands
2. **Colour-colour diagrams** showing red sequence / blue cloud bimodality
3. **Colour-redshift relations** confirming the 4000Å break tracking
4. **Photo-z evaluation** (when available)
5. **Colour-shape correlation** — red galaxies are rounder (intrinsic alignments)

### Project deliverables completed:
- Galaxy shape extraction (e1, e2 from REGAUSS)
- PSF model validation
- Shear calibration (m, c bias measurement)
- Multi-band photometry and colours
- Photo-z connection

These are the core skills for weak lensing with the LSST Science Pipelines.